# Tahap 1: Membangun Case Base (Versi Manual)
**Mata Kuliah:** Penalaran Komputer — SubCPMK-3 Case-Based Reasoning  
**Domain:** Perlindungan Anak  
**Tujuan:** Konversi dan preprocessing PDF putusan yang sudah diunduh manual

### Asumsi:
- PDF sudah tersedia di folder `data/pdf/`
- Penamaan file bebas (akan di-rename otomatis ke `case_001.pdf`, dst)

### Alur Notebook:
1. Install & Import
2. Konfigurasi & Setup Direktori
3. Setup Logging
4. Inventarisasi PDF yang Tersedia
5. Konversi PDF → Plain Text
6. Cleaning & Normalisasi Teks
7. Pipeline Utama
8. Validasi Akhir & Laporan
9. Preview Hasil

## Cell 1 — Install Dependencies

In [1]:
!pip install -q pdfminer.six tqdm

## Cell 2 — Import Library

In [2]:
import re
import json
import shutil
import logging
import unicodedata
from pathlib import Path
from tqdm import tqdm
from pdfminer.high_level import extract_text
from pdfminer.layout import LAParams

print("✅ Library berhasil diimport")

✅ Library berhasil diimport


## Cell 3 — Konfigurasi

In [5]:
CONFIG = {
    "JENIS_PERKARA" : "Perlindungan Anak",
    "MIN_DOKUMEN"   : 5,
    "MIN_KATA"      : 200,      # minimum kata agar dokumen dianggap valid
    "MIN_VALIDITAS" : 0.20,     # minimum rasio karakter bersih vs mentah
}

BASE_DIR = Path(".").resolve()
DATA_PDF = BASE_DIR / "data" / "pdf"
DATA_RAW = BASE_DIR / "data" / "raw"
LOGS_DIR = BASE_DIR / "logs"

for d in [DATA_PDF, DATA_RAW, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("📁 Direktori:")
print(f"   PDF input : {DATA_PDF}")
print(f"   Raw text  : {DATA_RAW}")
print(f"   Logs      : {LOGS_DIR}")
print()
print("⚠️  Pastikan file PDF sudah ada di folder data/pdf/ sebelum lanjut.")

📁 Direktori:
   PDF input : D:\PK\SUBCPMK3\STEP 1 MANUAL\data\pdf
   Raw text  : D:\PK\SUBCPMK3\STEP 1 MANUAL\data\raw
   Logs      : D:\PK\SUBCPMK3\STEP 1 MANUAL\logs

⚠️  Pastikan file PDF sudah ada di folder data/pdf/ sebelum lanjut.


## Cell 4 — Setup Logging

In [4]:
logging.basicConfig(
    level    = logging.INFO,
    format   = "%(asctime)s | %(levelname)s | %(message)s",
    handlers = [
        logging.FileHandler(LOGS_DIR / "cleaning.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)
logger.info(f"=== Sesi dimulai — Perkara: {CONFIG['JENIS_PERKARA']} ===")
print(f"📝 Log: {LOGS_DIR / 'cleaning.log'}")

2026-05-23 20:12:19,292 | INFO | === Sesi dimulai — Perkara: Perlindungan Anak ===


📝 Log: D:\PK\SUBCPMK3\STEP 1 MANUAL\logs\cleaning.log


## Cell 5 — Inventarisasi & Rename PDF

> Semua PDF di `data/pdf/` akan di-rename ke format `case_001.pdf`, `case_002.pdf`, dst.  
> Jika sudah bernama `case_XXX.pdf`, akan di-skip.

In [ ]:
def inventarisasi_pdf(folder: Path) -> list:
    """
    Scan folder, rename file ke format case_XXX.pdf jika belum,
    kembalikan list dict info per file.
    """
    semua_pdf = sorted(folder.glob("*.pdf"))

    if not semua_pdf:
        print(f"❌ Tidak ada file PDF di {folder}")
        print(f"   Salin file PDF ke folder tersebut lalu jalankan ulang cell ini.")
        return []

    hasil = []
    counter = 1

    for path_lama in semua_pdf:
        # Cek apakah sudah format case_XXX
        if re.match(r"^case_\d{3}\.pdf$", path_lama.name):
            case_id  = path_lama.stem          # e.g. case_001
            path_pdf = path_lama
        else:
            # Rename ke case_XXX.pdf
            case_id  = f"case_{counter:03d}"
            path_pdf = folder / f"{case_id}.pdf"

            # Hindari overwrite jika case_XXX sudah ada
            while path_pdf.exists():
                counter += 1
                case_id  = f"case_{counter:03d}"
                path_pdf = folder / f"{case_id}.pdf"

            shutil.copy2(str(path_lama), str(path_pdf))
            logger.info(f"Rename: {path_lama.name} → {path_pdf.name}")

        ukuran_kb = path_pdf.stat().st_size / 1024
        hasil.append({
            "case_id"      : case_id,
            "nama_asli"    : path_lama.name,
            "path_pdf"     : str(path_pdf),
            "ukuran_kb"    : round(ukuran_kb, 1),
            "status_pdf"   : "berhasil" if ukuran_kb > 5 else "terlalu_kecil",
        })
        counter += 1

    return hasil


daftar_pdf = inventarisasi_pdf(DATA_PDF)

print(f"\n📊 Inventarisasi PDF:")
print(f"   Total ditemukan : {len(daftar_pdf)}")
print(f"   Minimum required: {CONFIG['MIN_DOKUMEN']}")

valid   = [d for d in daftar_pdf if d["status_pdf"] == "berhasil"]
kecil   = [d for d in daftar_pdf if d["status_pdf"] == "terlalu_kecil"]

print(f"   ✅ Valid (>5KB)  : {len(valid)}")
if kecil:
    print(f"   ⚠️  Terlalu kecil : {len(kecil)}")
    for k in kecil:
        print(f"      {k['case_id']} ({k['ukuran_kb']} KB) — mungkin corrupt")

print(f"\n   Daftar PDF (5 pertama):")
for d in daftar_pdf[:5]:
    print(f"   [{d['case_id']}] {d['nama_asli'][:50]}  ({d['ukuran_kb']} KB)")

if len(valid) >= CONFIG["MIN_DOKUMEN"]:
    print(f"\n✅ {len(valid)} PDF valid — siap diproses!")
else:
    kekurangan = CONFIG["MIN_DOKUMEN"] - len(valid)
    print(f"\n⚠️  Kurang {kekurangan} PDF lagi dari minimum {CONFIG['MIN_DOKUMEN']}.")

# Simpan metadata awal
meta_file = BASE_DIR / "data" / "metadata_with_pdf.json"
meta_file.parent.mkdir(parents=True, exist_ok=True)
with open(meta_file, "w", encoding="utf-8") as f:
    json.dump(daftar_pdf, f, ensure_ascii=False, indent=2)
print(f"\n💾 Metadata: {meta_file}")

## Cell 6 — Fungsi Konversi PDF → Plain Text

In [ ]:
def pdf_ke_teks(path_pdf: Path) -> str | None:
    """
    Konversi PDF ke plain text via pdfminer.
    Returns string teks atau None jika gagal/kosong.
    """
    try:
        laparams = LAParams(
            line_overlap    = 0.5,
            char_margin     = 2.0,
            line_margin     = 0.5,
            word_margin     = 0.1,
            boxes_flow      = 0.5,
            detect_vertical = False,
        )
        teks = extract_text(str(path_pdf), laparams=laparams)
        return teks if teks and len(teks.strip()) > 50 else None
    except Exception as e:
        logger.error(f"Gagal ekstrak teks dari {path_pdf.name}: {e}")
        return None

print("✅ Fungsi konversi siap")

## Cell 7 — Fungsi Cleaning & Normalisasi

In [ ]:
class PembersihanTeks:
    """Membersihkan dan menormalisasi teks putusan MA RI."""

    # Pola header/footer/watermark yang sering muncul di putusan MA
    POLA_HAPUS = [
        r"^\s*\d+\s*$",
        r"halaman\s+\d+\s+(dari|of)\s+\d+",
        r"mahkamah\s+agung\s+republik\s+indonesia",
        r"direktori\s+putusan",
        r"putusan\.mahkamahagung\.go\.id",
        r"disclaimer\s*:.*",
        r"kepaniteraan\s+mahkamah\s+agung.*",
        r"salinan\s+putusan.*",
        r"\bwww\..*\.go\.id\b",
    ]

    def bersihkan(self, teks: str) -> str:
        """
        Pipeline pembersihan:
        1. Normalisasi Unicode
        2. Hapus karakter non-printable
        3. Hapus header/footer/watermark
        4. Normalisasi spasi & baris kosong
        """
        if not teks:
            return ""

        # 1. Unicode normalization
        teks = unicodedata.normalize("NFKC", teks)

        # 2. Hapus karakter non-printable
        teks = re.sub(
            r"[^\x09\x0A\x0D\x20-\x7E\x80-\xFF\u0100-\uFFFF]",
            " ", teks
        )

        # 3. Hapus baris yang mengandung header/footer
        baris_bersih = []
        for baris in teks.split("\n"):
            baris_lower = baris.lower().strip()
            if not any(re.search(p, baris_lower) for p in self.POLA_HAPUS):
                baris_bersih.append(baris)
        teks = "\n".join(baris_bersih)

        # 4. Normalisasi spasi
        teks = re.sub(r"[ \t]+",  " ",  teks)
        teks = re.sub(r"\n{3,}", "\n\n", teks)
        teks = re.sub(r"\n\s+\n", "\n\n", teks)

        return teks.strip()

    def hitung_kata(self, teks: str) -> int:
        return len(teks.split()) if teks else 0

    def hitung_validitas(self, teks_asli: str, teks_bersih: str) -> float:
        """
        Rasio karakter (bukan kata) — lebih stabil untuk PDF MA RI
        yang teks mentahnya sering penuh artefak spasi.
        """
        kar_asli   = len(teks_asli.replace(" ","").replace("\n",""))
        kar_bersih = len(teks_bersih.replace(" ","").replace("\n",""))
        if kar_asli == 0:
            return 0.0
        return kar_bersih / kar_asli


pembersih = PembersihanTeks()
print("✅ Kelas PembersihanTeks siap")

## Cell 8 — Pipeline Utama: PDF → Teks Bersih → Validasi → Simpan

In [ ]:
def pipeline_tahap1(daftar: list) -> dict:
    """
    Pipeline lengkap:
    PDF → Teks mentah → Cleaning → Validasi → Simpan ke data/raw/
    """
    stats = {
        "total"          : 0,
        "berhasil"       : 0,
        "gagal_ekstrak"  : 0,
        "gagal_validasi" : 0,
        "detail"         : [],
    }

    # Hanya proses yang PDF-nya valid
    valid = [d for d in daftar if d.get("status_pdf") == "berhasil"]
    print(f"🔄 Memproses {len(valid)} PDF...\n")

    for item in tqdm(valid, desc="Pipeline"):
        case_id  = item["case_id"]
        path_pdf = Path(item["path_pdf"])
        path_txt = DATA_RAW / f"{case_id}.txt"
        stats["total"] += 1

        info = {
            "case_id"    : case_id,
            "path_txt"   : str(path_txt),
            "jumlah_kata": 0,
            "validitas"  : 0.0,
            "status"     : "",
        }

        # Skip jika teks sudah ada
        if path_txt.exists() and path_txt.stat().st_size > 100:
            teks_ada = path_txt.read_text(encoding="utf-8", errors="ignore")
            info["jumlah_kata"] = pembersih.hitung_kata(teks_ada)
            info["status"]      = "sudah_ada"
            stats["berhasil"]  += 1
            logger.info(f"{case_id}: teks sudah ada, skip")
            stats["detail"].append(info)
            continue

        # Konversi PDF → teks mentah
        teks_mentah = pdf_ke_teks(path_pdf)
        if not teks_mentah:
            info["status"] = "gagal_ekstrak"
            stats["gagal_ekstrak"] += 1
            logger.error(f"{case_id}: ekstraksi teks gagal")
            stats["detail"].append(info)
            continue

        # Cleaning
        teks_bersih = pembersih.bersihkan(teks_mentah)
        jumlah_kata = pembersih.hitung_kata(teks_bersih)
        validitas   = pembersih.hitung_validitas(teks_mentah, teks_bersih)

        info["jumlah_kata"] = jumlah_kata
        info["validitas"]   = round(validitas, 3)

        # Validasi jumlah kata
        if jumlah_kata < CONFIG["MIN_KATA"]:
            info["status"] = "gagal_validasi_kata"
            stats["gagal_validasi"] += 1
            logger.warning(f"{case_id}: hanya {jumlah_kata} kata (min {CONFIG['MIN_KATA']})")
            stats["detail"].append(info)
            continue

        # Validasi rasio karakter
        if validitas < CONFIG["MIN_VALIDITAS"]:
            info["status"] = "gagal_validasi_rasio"
            stats["gagal_validasi"] += 1
            logger.warning(f"{case_id}: validitas {validitas:.2%} < {CONFIG['MIN_VALIDITAS']:.0%}")
            stats["detail"].append(info)
            continue

        # Simpan teks bersih
        path_txt.write_text(teks_bersih, encoding="utf-8")
        info["status"] = "berhasil"
        stats["berhasil"] += 1
        logger.info(f"{case_id}: ✅ {jumlah_kata} kata, validitas={validitas:.2%}")
        stats["detail"].append(info)

    return stats

print("✅ Fungsi pipeline_tahap1 siap")

In [ ]:
# Jalankan pipeline
statistik = pipeline_tahap1(daftar_pdf)

# Update metadata dengan path_txt
detail_map = {d["case_id"]: d for d in statistik["detail"]}
for item in daftar_pdf:
    cid = item["case_id"]
    if cid in detail_map:
        item["path_txt"]    = detail_map[cid]["path_txt"]
        item["status_teks"] = detail_map[cid]["status"]
        item["jumlah_kata"] = detail_map[cid]["jumlah_kata"]

# Simpan metadata yang sudah diperbarui
with open(BASE_DIR / "data" / "metadata_with_pdf.json", "w", encoding="utf-8") as f:
    json.dump(daftar_pdf, f, ensure_ascii=False, indent=2)

# Simpan statistik
with open(LOGS_DIR / "stats_tahap1.json", "w", encoding="utf-8") as f:
    json.dump(statistik, f, ensure_ascii=False, indent=2)

print(f"\n📊 Statistik Pipeline:")
print(f"   Total diproses   : {statistik['total']}")
print(f"   ✅ Berhasil       : {statistik['berhasil']}")
print(f"   ❌ Gagal ekstrak  : {statistik['gagal_ekstrak']}")
print(f"   ⚠️  Gagal validasi : {statistik['gagal_validasi']}")

## Cell 9 — Validasi Akhir & Laporan

In [ ]:
file_txt   = sorted(DATA_RAW.glob("*.txt"))
total      = len(file_txt)

print(f"{'='*55}")
print(f"  LAPORAN VALIDASI AKHIR — TAHAP 1")
print(f"{'='*55}")
print(f"  Total file .txt  : {total}")
print(f"  Minimum required : {CONFIG['MIN_DOKUMEN']}")

if total >= CONFIG["MIN_DOKUMEN"]:
    print(f"  Status           : ✅ MEMENUHI KUOTA")
    logger.info(f"Validasi akhir: LULUS — {total} dokumen")
else:
    kekurangan = CONFIG["MIN_DOKUMEN"] - total
    print(f"  Status           : ❌ KURANG {kekurangan} dokumen")
    print(f"  Saran            : Tambah PDF ke data/pdf/ lalu jalankan ulang")
    logger.warning(f"Validasi akhir: KURANG {kekurangan} dokumen")

if file_txt:
    kata_list = [
        len(f.read_text(encoding="utf-8", errors="ignore").split())
        for f in file_txt
    ]
    print(f"\n  Statistik jumlah kata per dokumen:")
    print(f"    Min  : {min(kata_list):,}")
    print(f"    Max  : {max(kata_list):,}")
    print(f"    Rata : {sum(kata_list) // len(kata_list):,}")

    # Dokumen di bawah threshold kata
    pendek = [(f.name, k) for f, k in zip(file_txt, kata_list)
              if k < CONFIG["MIN_KATA"] * 2]
    if pendek:
        print(f"\n  ⚠️  Dokumen dengan kata < {CONFIG['MIN_KATA']*2} (perlu dicek manual):")
        for nama, k in pendek:
            print(f"     {nama}: {k} kata")

print(f"{'='*55}")
print(f"  Log  : {LOGS_DIR / 'cleaning.log'}")
print(f"  Stats: {LOGS_DIR / 'stats_tahap1.json'}")
print(f"{'='*55}")

## Cell 10 — Preview Hasil Cleaning

In [ ]:
file_txt = sorted(DATA_RAW.glob("*.txt"))

if not file_txt:
    print("⚠️  Belum ada file .txt — jalankan pipeline terlebih dahulu")
else:
    print(f"📄 Menampilkan preview {min(3, len(file_txt))} dokumen pertama:\n")
    for f in file_txt[:3]:
        teks = f.read_text(encoding="utf-8", errors="ignore")
        kata = len(teks.split())
        print(f"{'─'*55}")
        print(f"  File    : {f.name}")
        print(f"  Kata    : {kata:,}")
        print(f"  Preview :")
        print()
        print(teks[:400])
        print(f"  ...")
        print()

---
## ✅ Tahap 1 Selesai!

**Struktur output:**
```
/data/
├── pdf/
│   ├── case_001.pdf
│   ├── case_002.pdf
│   └── ...
├── raw/
│   ├── case_001.txt   ← teks bersih siap pakai
│   ├── case_002.txt
│   └── ...
└── metadata_with_pdf.json
/logs/
├── cleaning.log
└── stats_tahap1.json
```

**Langkah selanjutnya:** Buka `02_case_representation.ipynb`